# **EDA: El impacto de la IA**
## Memoria del proyecto

Proyecto del módulo Data Analysis: análisis exploratorio de datos (EDA).

## 1. Introducción

La IA llegó para quedarse y ya lleva tiempo en nuestras vidas. Forma parte del trabajo y del estudio. Se da por hecho que nos hace más productivos, pero ¿realmente es así? ¿Tiene efectos en el bienestar y el aprendizaje? Este proyecto explora su impacto con datos.

**Pregunta principal:** ¿Usar IA nos hace más productivos? ¿A qué precio?

**Hipótesis** (planteadas antes de analizar los datos):

1. Quien usa más IA es más productivo.
2. El uso excesivo de la IA tiene efectos negativos en el bienestar.
3. Los estudiantes que usan más IA sacan mejores notas, pero retienen menos lo aprendido.

**Cómo está organizado el trabajo.** Esta memoria resume los pasos del análisis. El detalle completo está en los notebooks de `src/notebooks/`:

| Notebook | Contenido |
|---|---|
| `EDA_Impacto_de_la_IA_limpieza.ipynb` | Unión de ficheros, revisión de la calidad de los datos y decisiones de limpieza |
| `EDA_Impacto_de_la_IA_analisis.ipynb` | Comprobación de cada hipótesis, paso a paso |
| `EDA_Impacto_de_la_IA_visualizaciones.ipynb` | Gráficos de cada hipótesis, qué muestra cada uno y por qué se eligieron |

Las funciones y variables están en `src/utils/` y el script `main.py` genera las imágenes.

## 2. Fuentes de datos

Los datos vienen de Kaggle. Son tres ficheros que forman dos datasets:

| Dataset | Fichero(s) | Tamaño | Qué contiene |
|---|---|---|---|
| Empleados | `ai_productivity_features.csv` y `ai_productivity_targets.csv` | 4.500 filas y 17 columnas (15 + 2) | Una fila por empleado (datos mensuales simulados). `features`: puesto, experiencia, horas de uso de IA, tareas automatizadas, trabajo manual, reuniones, colaboración, foco, presión de plazos y equilibrio vida-trabajo. `targets`: productividad y riesgo de burnout. Se unen por `Employee_ID`. |
| Estudiantes | `ai_student_impact_dataset.csv` | 50.000 filas y 16 columnas | Una fila por estudiante universitario: uso de IA (horas semanales, tipo de uso, nivel con los prompts), horas de estudio tradicional, nota media antes y después del semestre, retención de habilidades, dependencia percibida, ansiedad en los exámenes y burnout. |

Cada hipótesis usa un dataset: la 1 (productividad) el de empleados, la 3 (notas y retención) el de estudiantes y la 2 (bienestar) los dos.

**Los datos son sintéticos**. Es decir, generados artificialmente. Por eso las conclusiones describen lo que ocurre en un escenario simulado, no evidencia sobre personas reales.

In [2]:
import pandas as pd

# Empleados: se unen los dos ficheros por Employee_ID
df_features = pd.read_csv("../data/ai_productivity_features.csv")
df_targets = pd.read_csv("../data/ai_productivity_targets.csv")
df_empleados = pd.merge(df_features, df_targets, on="Employee_ID")

# Estudiantes
df_estudiantes = pd.read_csv("../data/ai_student_impact_dataset.csv")

print("Empleados:", df_empleados.shape)
print("Estudiantes:", df_estudiantes.shape)

Empleados: (4500, 17)
Estudiantes: (50000, 16)


## 3. Limpieza y calidad de los datos

Como los datasets son sintéticos, además de buscar errores se revisa cómo se han generado los datos. Detalle completo en el notebook de limpieza.

### 3.1 Nulos y duplicados

In [3]:
for nombre, df in [("Empleados", df_empleados), ("Estudiantes", df_estudiantes)]:
    print(nombre, "-> nulos:", df.isna().sum().sum(), "| duplicados:", df.duplicated().sum())

Empleados -> nulos: 0 | duplicados: 0
Estudiantes -> nulos: 0 | duplicados: 0


No hay nulos ni duplicados en ninguno de los dos datasets, así que no se elimina ninguna fila. Las 4.500 filas de empleados salen de unir los dos ficheros sin perder ni repetir a nadie.

### 3.2 Rangos y valores en los límites

Con `describe()` se buscan valores imposibles (no hay ninguno) y variables con un tope o un suelo. Cuando una variable tiene decimales, es poco habitual que mucha gente repita exactamente el mismo valor extremo: si ocurre, el dato probablemente se calculó por encima del límite y luego se cortó. Se mide qué porcentaje de personas está justo en cada límite.

In [4]:
limites = pd.Series({
    "Empleados: burnout = 10": (df_empleados["burnout_risk_score"] == 10).mean() * 100,
    "Empleados: errores = 0,10": (df_empleados["error_rate_percent"] == 0.10).mean() * 100,
    "Empleados: productividad = 100": (df_empleados["productivity_score"] == 100).mean() * 100,
    "Empleados: equilibrio vida-trabajo = 1": (df_empleados["work_life_balance_score"] == 1).mean() * 100,
    "Estudiantes: nota final = 4,0": (df_estudiantes["Post_Semester_GPA"] == 4).mean() * 100,
    "Estudiantes: retención = 100": (df_estudiantes["Skill_Retention_Score"] == 100).mean() * 100,
    "Estudiantes: horas de IA = 40": (df_estudiantes["Weekly_GenAI_Hours"] == 40).mean() * 100,
    "Estudiantes: horas de IA = 0": (df_estudiantes["Weekly_GenAI_Hours"] == 0).mean() * 100,
})
limites.round(2).rename("% de personas")

Empleados: burnout = 10                   37.44
Empleados: errores = 0,10                 14.47
Empleados: productividad = 100             0.73
Empleados: equilibrio vida-trabajo = 1     4.29
Estudiantes: nota final = 4,0              9.61
Estudiantes: retención = 100               4.29
Estudiantes: horas de IA = 40              0.90
Estudiantes: horas de IA = 0               0.06
Name: % de personas, dtype: float64

| Variable | Resultado | Interpretación |
|---|---|---|
| Burnout de empleados = 10 | 37,4 % | Tope claro. Limita la hipótesis 2. |
| Errores de empleados = 0,10 | 14,5 % | Suelo. Parece un mínimo artificial. |
| Nota final = 4,0 | 9,6 % | Casi 1 de cada 10 no puede subir más. Afecta a la hipótesis 3. |
| Retención = 100 | 4,3 % | Tope leve. |
| Horas de IA = 40 | 0,9 % | Tope leve. |
| Horas de IA = 0 | 0,06 % | Casi todos los estudiantes usan IA: se compara uso bajo frente a alto, no usuarios frente a no usuarios. |
| Productividad = 100 y equilibrio vida-trabajo = 1 | 0,7 % y 4,3 % | Sin problema. |

Otras observaciones de `describe()`: `Student_ID` es un identificador consecutivo (de 100001 a 150000), así que no se usa en el análisis.
 
En empleados el trabajo manual, la colaboración y las horas de foco tienen mínimos y máximos redondos, por lo que probablemente están acotados.

### 3.3 Categorías

Se cuenta cuántas personas hay en cada categoría para saber si los grupos son comparables. Los grupos de `job_role` (de 1.115 developers a 461 writers) y los de `Primary_Use_Case` en estudiantes (el menor tiene 6.340) son lo bastante grandes. En cambio, el burnout de empleados está muy desequilibrado: 73 % High, 24 % Medium y solo 110 empleados (2,4 %) en Low. En estudiantes está bien repartido (33 % Low, 42 % Medium y 25 % High).

Los empleados tienen dos medidas de burnout: `burnout_risk_score` (puntuación) y `burnout_risk_level` (nivel). Se comprueba si son la misma información.

In [5]:
df_empleados.groupby("burnout_risk_level")["burnout_risk_score"].agg(["count", "min", "max"])

,count,min,max
burnout_risk_level,,,
High,3303,7.01,10.00
Low,110,3.22,4.49
Medium,1087,4.54,7.00


Los rangos no se solapan: el nivel es la puntuación dividida en tres tramos. Es la misma información y no cuentan como dos pruebas distintas.

### 3.4 Decisiones de limpieza

- No se elimina ninguna fila (no hay nulos ni duplicados).
- Los valores en los límites no se consideran errores: parecen límites de cómo se generó el dato. Si se eliminaran, se sesgaría el resultado. Se mantienen y se documentan con su porcentaje.
- Se usa `burnout_risk_score` como medida principal de burnout en empleados porque es más detallada. `burnout_risk_level` solo se usa para presentar grupos.
- Se descarta `Student_ID` del análisis porque es solo un identificador.
- **Límites que se tienen en cuenta en el análisis:** 
    - El burnout de empleados está cortado en 10 (poca variación para la hipótesis 2). 
    - El equilibrio vida-trabajo sirve como segunda medida de bienestar. 
    - El 9,6 % de estudiantes termina con un 4,0, efecto techo en la hipótesis 3.
    - La productividad está bien repartida, así que la hipótesis 1 no tiene este problema.

## 4. Desarrollo del análisis

Detalle completo en el notebook de análisis.

### 4.1 Tramos de uso de IA

Para ver cómo cambian los resultados al usar más IA, se divide a las personas en 4 grupos de tamaño parecido según sus horas semanales de IA (cuartiles, con `pd.qcut`). Se añade la columna `tramo_ia`. Cada dataset tiene sus propios cortes.

In [6]:
etiquetas = ["Bajo", "Medio-bajo", "Medio-alto", "Alto"]

df_empleados["tramo_ia"] = pd.qcut(df_empleados["ai_tool_usage_hours_per_week"], q=4, labels=etiquetas)
df_estudiantes["tramo_ia"] = pd.qcut(df_estudiantes["Weekly_GenAI_Hours"], q=4, labels=etiquetas)

print("Empleados (horas de IA a la semana)")
print(df_empleados.groupby("tramo_ia", observed=True)["ai_tool_usage_hours_per_week"].agg(["min", "max", "count"]))
print()
print("Estudiantes (horas de IA a la semana)")
print(df_estudiantes.groupby("tramo_ia", observed=True)["Weekly_GenAI_Hours"].agg(["min", "max", "count"]))

Empleados (horas de IA a la semana)
             min   max  count
tramo_ia                     
Bajo         0.0   6.0   1143
Medio-bajo   6.1   9.7   1110
Medio-alto   9.8  14.3   1144
Alto        14.4  30.4   1103

Estudiantes (horas de IA a la semana)
              min    max  count
tramo_ia                       
Bajo         0.00   2.39  12510
Medio-bajo   2.40   5.80  12516
Medio-alto   5.81  11.72  12479
Alto        11.73  40.00  12495


Los tramos no son exactamente iguales en empleados (de 1.103 a 1.144 personas en lugar de 1.125) porque hay personas con justo las horas de los cortes y `qcut` no puede separarlas. Los tamaños son parecidos, así que se pueden comparar.

### 4.2 Hipótesis 1: Quien usa más IA es más productivo

**Paso 1. Correlación.** La correlación (de -1 a 1) mide si dos variables suben y bajan a la vez. Se calcula la de todas las variables numéricas con la productividad.

In [7]:
df_empleados.corr(numeric_only=True)["productivity_score"].sort_values().round(2)

meeting_hours_per_week         -0.45
manual_work_hours_per_week     -0.44
burnout_risk_score             -0.39
collaboration_hours_per_week   -0.36
error_rate_percent             -0.30
task_complexity_score           0.01
learning_time_hours_per_week    0.11
work_life_balance_score         0.40
experience_years                0.46
ai_tool_usage_hours_per_week    0.50
focus_hours_per_day             0.51
tasks_automated_percent         0.53
productivity_score              1.00
Name: productivity_score, dtype: float64

- Horas de IA y productividad: 0,50. Relación positiva y moderada, no exacta.
- Las horas de IA no son lo que más se relaciona con la productividad. Las tareas automatizadas (0,53) y las horas de foco (0,51) tienen valores parecidos o mayores. Las reuniones (-0,45) y el trabajo manual (-0,44) van en sentido contrario.
- Las tareas automatizadas se mueven casi a la vez que las horas de IA (0,95), miden prácticamente lo mismo. No son una prueba independiente.
- Quienes usan más IA dedican menos horas al trabajo manual (-0,63) y a reuniones (-0,27), lo que también puede influir en su productividad.

**Paso 2. Productividad por tramo.** Se compara la productividad media de los cuatro grupos.

In [8]:
resumen_prod = df_empleados.groupby("tramo_ia", observed=True)["productivity_score"].agg(["mean", "count"])
medias_prod = resumen_prod["mean"]

dif_prod = medias_prod.loc["Alto"] - medias_prod.loc["Bajo"]
pct_prod = dif_prod / medias_prod.loc["Bajo"] * 100

print(resumen_prod.round(1))
print()
print("Diferencia Alto - Bajo:", round(dif_prod, 1), "puntos (", round(pct_prod, 1), "% más )")

            mean  count
tramo_ia               
Bajo        55.9   1143
Medio-bajo  62.1   1110
Medio-alto  67.5   1144
Alto        74.6   1103

Diferencia Alto - Bajo: 18.7 puntos ( 33.5 % más )


La productividad sube en cada tramo: 55,9, 62,1, 67,5 y 74,6. Entre el grupo que menos usa IA y el que más hay **18,7 puntos de diferencia** (un 33,5 % más).

**Paso 3. ¿Se mantiene dentro de cada puesto?** Los puestos usan la IA de forma distinta (Developer y Writer unas 15 h a la semana, Manager y Designer unas 5 h) y el Manager es el menos productivo. Por eso la relación anterior podría deberse al puesto. Para comprobarlo se compara dentro de cada puesto y se cuenta cuántas personas hay en cada celda porque una media de pocas personas no es fiable.

In [9]:
pivot = df_empleados.pivot_table(index="job_role", columns="tramo_ia", values="productivity_score",
                                 aggfunc="mean", observed=True).round(1)
conteo = pd.crosstab(df_empleados["job_role"], df_empleados["tramo_ia"])

print("Productividad media por puesto y tramo")
print(pivot)
print()
print("Personas por puesto y tramo")
print(conteo)

Productividad media por puesto y tramo
tramo_ia   Bajo  Medio-bajo  Medio-alto  Alto
job_role                                     
Analyst    59.2        63.3        67.4  74.4
Designer   59.6        64.2        64.5  81.5
Developer  62.2        61.9        68.1  75.2
Manager    49.1        55.1        56.0   NaN
Marketer   59.8        64.1        68.2  71.8
Writer     65.0        64.0        69.0  74.1

Personas por puesto y tramo
tramo_ia   Bajo  Medio-bajo  Medio-alto  Alto
job_role                                     
Analyst     153         288         341   110
Designer    435         246          40     1
Developer    31         119         323   642
Manager     418         195          39     0
Marketer     94         211         265    88
Writer       12          51         136   262


- En Analyst, Developer, Marketer y Writer, el grupo Alto es más productivo que el Bajo (por ejemplo, Analyst pasa de 59,2 a 74,4 y Developer de 62,2 a 75,2). La relación se mantiene dentro del puesto y no se explica solo por él.
- En Manager no hay nadie en el tramo Alto. El `NaN` no es un dato perdido: el máximo de un manager es 13,5 h y el corte está en 14,3. Se compara hasta Medio-alto (de 49,1 a 56,0).
- En Designer solo hay 1 persona en Alto, así que ese dato no se interpreta.
- En Developer (31 personas en Bajo) y Writer (12) el primer escalón no sube, pero son grupos muy pequeños.

**Paso 4. ¿A costa de errores?** Se calcula el porcentaje medio de errores por tramo. (La ficha de Kaggle no define esta variable. Se interpreta como el porcentaje de errores en el trabajo).

In [10]:
medias_err = df_empleados.groupby("tramo_ia", observed=True)["error_rate_percent"].mean()
dif_err = medias_err["Alto"] - medias_err["Bajo"]
pct_err = dif_err / medias_err["Bajo"] * 100

print(medias_err.round(2))
print()
print("Diferencia Alto - Bajo:", round(dif_err, 2), "puntos (", round(pct_err, 1), "% más )")

tramo_ia
Bajo          1.80
Medio-bajo    1.99
Medio-alto    2.14
Alto          2.47
Name: error_rate_percent, dtype: float64

Diferencia Alto - Bajo: 0.67 puntos ( 37.0 % más )


Los errores pasan del 1,80 % al 2,47 %: 0,67 puntos más, un 37 % más y aumentan en cada tramo.

**Conclusión de la hipótesis 1: Se cumple.** La productividad media sube en cada tramo de uso de IA (de 55,9 a 74,6) y la subida se mantiene dentro de los puestos con datos suficientes. Tiene un coste: el grupo que más IA usa también tiene más errores (2,47 % frente a 1,80 %), aunque sigue siendo el más productivo.

*Limitaciones:* es una asociación, no prueba que la IA cause la mejora (quienes usan más IA también dedican menos tiempo a trabajo manual y reuniones) y los datos son sintéticos.

### 4.3 Hipótesis 2: el uso excesivo de la IA tiene efectos negativos en el bienestar

El bienestar se mide en los dos datasets. El uso excesivo es el tramo Alto. En empleados se miran `burnout_risk_score` y `work_life_balance_score`. En estudiantes, el burnout y la ansiedad.

**Empleados.** Se comparan las horas de IA con el bienestar y, para saber qué pesa más, con la presión de plazos.

In [11]:
print("Correlación con las horas de IA:")
print(df_empleados[["ai_tool_usage_hours_per_week", "burnout_risk_score", "work_life_balance_score"]].corr().round(3).iloc[0])
print()
print("Bienestar medio por tramo de IA:")
print(df_empleados.groupby("tramo_ia", observed=True)[["burnout_risk_score", "work_life_balance_score"]].mean().round(2))
print()
print("Bienestar y horas de IA medias según la presión de plazos:")
print(df_empleados.groupby("deadline_pressure_level")[["burnout_risk_score", "work_life_balance_score",
                                                       "ai_tool_usage_hours_per_week"]].mean().round(2))

Correlación con las horas de IA:
ai_tool_usage_hours_per_week    1.000
burnout_risk_score              0.038
work_life_balance_score        -0.063
Name: ai_tool_usage_hours_per_week, dtype: float64

Bienestar medio por tramo de IA:
            burnout_risk_score  work_life_balance_score
tramo_ia                                               
Bajo                      8.34                     4.75
Medio-bajo                8.30                     4.79
Medio-alto                8.34                     4.84
Alto                      8.45                     4.51

Bienestar y horas de IA medias según la presión de plazos:
                         burnout_risk_score  work_life_balance_score  \
deadline_pressure_level                                                
High                                   9.97                     3.29   
Low                                    6.27                     6.17   
Medium                                 8.87                     4.69   

           

- Entre el tramo Bajo y el Alto el burnout medio apenas cambia (de 8,34 a 8,45) y su correlación con las horas de IA es de 0,038, prácticamente nula. El porcentaje de empleados con burnout alto tampoco sigue una tendencia clara (73,8 %, 71,8 %, 72,6 % y 75,4 %).
- La presión de plazos sí marca diferencias: El burnout medio es de 6,27 con presión baja, 8,87 con media y 9,97 con alta, mientras que las horas de IA son casi iguales en los tres niveles (unas 10,3 h).
- La variable que más se relaciona con el burnout es el equilibrio vida-trabajo (-0,85), muy por delante de las horas de IA (0,04).

**Estudiantes.** El burnout es una categoría (Low, Medium, High), así que se miran los porcentajes por tramo. La ansiedad en los exámenes va de 1 a 10.

In [12]:
print("% de estudiantes por nivel de burnout en cada tramo:")
print((pd.crosstab(df_estudiantes["tramo_ia"], df_estudiantes["Burnout_Risk_Level"], normalize="index") * 100).round(2))
print()
print("Ansiedad media en los exámenes por tramo:")
print(df_estudiantes.groupby("tramo_ia", observed=True)["Anxiety_Level_During_Exams"].mean().round(2))

% de estudiantes por nivel de burnout en cada tramo:
Burnout_Risk_Level   High    Low  Medium
tramo_ia                                
Bajo                 9.06  49.66   41.27
Medio-bajo          12.93  41.91   45.16
Medio-alto          22.16  29.87   47.98
Alto                55.78   9.47   34.75

Ansiedad media en los exámenes por tramo:
tramo_ia
Bajo          3.81
Medio-bajo    3.90
Medio-alto    4.21
Alto          5.16
Name: Anxiety_Level_During_Exams, dtype: float64


El burnout alto pasa del 9,06 % en el tramo Bajo al 12,93 %, 22,16 % y 55,78 % en el Alto. El burnout bajo cae del 49,66 % al 9,47 % y la ansiedad sube de 3,81 a 5,16. El salto grande está en el tramo Alto (más de 11,73 h a la semana).

**Conclusión de la hipótesis 2: Se cumple solo en estudiantes.** En estudiantes el uso excesivo se asocia con peor bienestar. 

En empleados el burnout casi no cambia con las horas de IA y se relaciona con la presión de plazos y el equilibrio vida-trabajo. 

Con estos datos no se puede saber por qué la tendencia es distinta en cada grupo.

*Limitaciones:* El burnout de empleados está cortado en 10 (37,4 % de los casos), lo que deja poca variación. Las escalas no son comparables (puntuación en empleados, categorías en estudiantes), así que solo se compara la tendencia. Correlación no es causalidad (no se sabe si usar más IA provoca burnout o si quien tiene más estrés usa más IA). 

Los datos son sintéticos. Además, la ficha de los estudiantes dice que variables como el burnout se generaron de forma uniforme, pero aquí cambia claramente con las horas de IA. La descripción no coincide del todo con los datos.

### 4.4 Hipótesis 3: Los estudiantes que usan más IA sacan mejores notas, pero retienen menos

La nota final por sí sola no dice si el alumno mejoró porque cada uno parte de un nivel. Se crea `mejora_nota` (nota final menos nota inicial) y se compara por tramo, junto con la retención de habilidades.

In [13]:
df_estudiantes["mejora_nota"] = df_estudiantes["Post_Semester_GPA"] - df_estudiantes["Pre_Semester_GPA"]

print("Correlación con las horas de IA:")
print(df_estudiantes[["Weekly_GenAI_Hours", "mejora_nota", "Skill_Retention_Score"]].corr().round(3).iloc[0])
print()
medias_estudiantes = df_estudiantes.groupby("tramo_ia", observed=True)[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)
print("Mejora de nota y retención medias por tramo:")
print(medias_estudiantes)

Correlación con las horas de IA:
Weekly_GenAI_Hours       1.000
mejora_nota             -0.046
Skill_Retention_Score   -0.118
Name: Weekly_GenAI_Hours, dtype: float64

Mejora de nota y retención medias por tramo:
            mejora_nota  Skill_Retention_Score
tramo_ia                                      
Bajo              0.188                 75.653
Medio-bajo        0.206                 76.429
Medio-alto        0.229                 77.117
Alto              0.190                 73.995


- La mejora media es de 0,20 puntos de GPA. Las correlaciones de las horas de IA con la mejora (-0,05) y con la retención (-0,12) son muy débiles, y la correlación solo detecta líneas rectas, así que se compara por grupos.
- La mejora sube del tramo Bajo al Medio-alto (0,188, 0,206 y 0,229) y baja en el Alto (0,190), casi igual que en el Bajo. La retención sigue el mismo patrón: de 75,65 sube a 77,12 y cae a 73,99 en el Alto (3,12 puntos menos que en el tramo anterior).
- Las diferencias son pequeñas: entre Bajo y Medio-alto la mejora es de 0,04 puntos sobre 4.

**Efecto techo.** El 9,6 % de los alumnos acaba con un 4,0, y quien ya parte de una nota alta apenas puede mejorar. Para comprobar si eso distorsiona los resultados, se repite la comparación sin el 25 % con mejor nota inicial (corte de 3,5).

In [14]:
sin_techo = df_estudiantes[df_estudiantes["Pre_Semester_GPA"] < 3.5]

print("Estudiantes que se quedan:", len(sin_techo), "(", round(len(sin_techo) / len(df_estudiantes) * 100, 1), "% )")
sin_techo.groupby("tramo_ia", observed=True)[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)

Estudiantes que se quedan: 36654 ( 73.3 % )


,mejora_nota,Skill_Retention_Score
tramo_ia,,
Bajo,0.201,75.181
Medio-bajo,0.218,75.814
Medio-alto,0.245,76.620
Alto,0.206,73.524


Sale el mismo patrón. La mejora pasa de 0,201 en Bajo a 0,245 en Medio-alto y 0,206 en Alto. La retención de 75,18 a 76,62 y 73,52. así que el efecto techo no parece explicar lo observado.

**¿Importa más cómo se usa?** Se mira si el tipo de uso (`Primary_Use_Case`) se asocia con más diferencias que las horas.

In [15]:
medias_uso = df_estudiantes.groupby("Primary_Use_Case")[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)

print(medias_uso)
print()
print("Distancia entre el mejor y el peor tipo de uso:")
print((medias_uso.max() - medias_uso.min()).round(3))
print()
print("Distancia entre el mejor y el peor tramo de horas:")
print((medias_estudiantes.max() - medias_estudiantes.min()).round(3))

                           mejora_nota  Skill_Retention_Score
Primary_Use_Case                                             
Copywriting/Drafting             0.200                 75.231
Debugging/Troubleshooting        0.249                 78.069
Direct_Answer_Generation         0.133                 73.719
Ideation                         0.200                 75.520
Summarizing_Reading              0.197                 75.225

Distancia entre el mejor y el peor tipo de uso:
mejora_nota              0.116
Skill_Retention_Score    4.350
dtype: float64

Distancia entre el mejor y el peor tramo de horas:
mejora_nota              0.041
Skill_Retention_Score    3.122
dtype: float64


Depurar código se asocia con la mayor mejora (0,249) y la mayor retención (78,07) y pedir respuestas directas con la menor (0,133 y 73,72). El tipo de uso se asocia con diferencias mayores que las horas (0,116 frente a 0,041 puntos de mejora).

**Conclusión de la hipótesis 3: Se cumple a medias.** El uso alto se asocia con menos retención (de 75,65 en Bajo a 73,99 en Alto), pero las notas no mejoran con más usO. La mejora es casi igual en Bajo y en Alto (0,188 y 0,190) y solo sube con un uso moderado. No se observa «mejores notas, pero menos aprendizaje. Donde más mejora la nota (uso moderado) también es mayor la retención. Importa más cómo se usa la IA que cuánto.

*Limitaciones:* La retención es solo un indicador del aprendizaje. Una correlación no demuestra causalidad y el análisis por tipo de uso no lo cruza con las horas. Según la ficha de Kaggle, la nota final se generó a partir de la nota inicial, el uso de IA y las horas de estudio, por lo que es esperable que aparezcan relaciones entre estas variables.

## 5. Visualizaciones

Los gráficos de cada hipótesis se construyen en el notebook de visualizaciones (con su explicación) y se guardan en `src/img/`. Estos son los seis resultados.

### Hipótesis 1

![Productividad y errores por tramo](img/h1_productividad_errores.png)

**Figura 1.** Productividad media (izquierda) y porcentaje medio de errores (derecha) por tramo de horas de IA. La productividad sube de 55,9 a 74,6 y los errores de 1,80 % a 2,47 %.

![Productividad por puesto y tramo](img/h1_heatmap_puesto.png)

**Figura 2.** Productividad media por puesto y tramo. La subida aparece dentro de cada puesto. Las celdas vacías son combinaciones con menos de 30 personas.

### Hipótesis 2

![Burnout en empleados](img/h2_burnout_empleados.png)

**Figura 3.** Burnout medio de empleados según las horas de IA (casi no cambia) y según la presión de plazos (sí cambia), con la misma escala.

![Burnout en estudiantes](img/h2_burnout_estudiantes.png)

**Figura 4.** Reparto del burnout de los estudiantes en cada tramo. El burnout alto pasa del 9,1 % al 55,8 %, con el salto grande en el tramo Alto.

### Hipótesis 3

![Mejora de nota y retención](img/h3_mejora_retencion.png)

**Figura 5.** Mejora de nota (izquierda) y retención de habilidades (derecha) por tramo. Con uso alto la nota no mejora más y cae la retención. El eje no empieza en 0 y las diferencias son pequeñas.

![Mejora de nota según el tipo de uso](img/h3_mejora_por_uso.png)

**Figura 6.** Mejora media de nota según el tipo de uso principal. Pedir respuestas directas es el uso que menos mejora la nota y depurar código el que más.

## 6. Conclusiones

| Hipótesis | Resultado | Evidencia principal |
|---|---|---|
| 1. Quien usa más IA es más productivo | **Se cumple** | La productividad media sube de 55,9 (tramo Bajo) a 74,6 (tramo Alto), con un coste: más errores (1,80 % a 2,47 %). |
| 2. El uso excesivo de la IA tiene efectos negativos en el bienestar | **Se cumple solo en estudiantes** | En estudiantes, el burnout alto pasa del 9,06 % al 55,78 %. En empleados, el burnout apenas cambia (8,34 a 8,45) y depende de la presión de plazos. |
| 3. Los estudiantes que usan más IA sacan mejores notas, pero retienen menos | **Se cumple a medias** | La retención cae solo con uso alto (75,65 a 73,99) y la mejora de nota es casi igual en Bajo y Alto (0,188 y 0,190). |

**Respuesta a la pregunta principal.** ¿Usar IA nos hace más productivos? En los datos de empleados, sí: quienes más la usan puntúan 18,7 puntos más en productividad (un 33,5 % más). ¿A qué precio? Cometen más errores (un 37 % más) y, en el caso de los estudiantes, un uso muy alto va acompañado de mucho más burnout y ansiedad y de menos retención de lo aprendido. En empleados, lo que se relaciona con el burnout no es la IA, sino la presión de plazos y el equilibrio vida-trabajo. Además, en estudiantes importa más para qué se usa la IA que cuántas horas.

**Limitaciones generales.**
- Los datos son sintéticos: Los resultados describen un escenario simulado, no personas reales.
- Todas las relaciones son asociaciones. Los datos no permiten saber si la IA causa los cambios.
- Los topes de algunas variables (burnout de empleados en 10, nota final en 4,0) reducen la variación disponible.
- Los dos datasets son poblaciones distintas y no se pueden cruzar fila a fila.

**Oportunidades.** Con la cautela de que los datos son simulados, el análisis apunta a que:
- En una empresa conviene vigilar la presión de plazos y el equilibrio vida-trabajo más que las horas de IA y revisar los errores cuando se extiende su uso.
- En educación, parece más útil orientar hacia usos como depurar o generar ideas que hacia pedir respuestas directas y vigilar el burnout de quienes la usan de forma intensiva.
- Como siguiente paso, repetir el análisis con datos reales y comprobar el efecto de la IA controlando por puesto, experiencia y carga de trabajo.